In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

pd.set_option("display.max_columns", None)

PROCESSED_DATA_DIR = Path("C:\\Users\\aabre\\Documents\\data-science-projects\\nba-player-archetype-clustering\\data\\processed\\2025_26")
FIGURES_DIR = Path("C:\\Users\\aabre\\Documents\\data-science-projects\\nba-player-archetype-clustering\\figures")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

players = pd.read_csv(PROCESSED_DATA_DIR / "STYLE_FEATURES_V1.csv")
players.info()

<class 'pandas.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   PLAYER_ID                450 non-null    int64  
 1   PLAYER_NAME              450 non-null    str    
 2   USG_RATE                 450 non-null    float64
 3   TOUCHES_PER_100          450 non-null    float64
 4   POT_AST_PER_100          450 non-null    float64
 5   DRIVES_PER_100           450 non-null    float64
 6   CATCH_SHOOT_FGA_PER_100  450 non-null    float64
 7   PULL_UP_FGA_PER_100      450 non-null    float64
 8   PAINT_TOUCHES_PER_100    450 non-null    float64
 9   SCREEN_AST_PER_100       450 non-null    float64
 10  DEFLECTIONS_PER_100      450 non-null    float64
 11  STL_PCT                  450 non-null    float64
 12  BLK_PCT                  450 non-null    float64
 13  BOXOUTS_PER_100          450 non-null    float64
 14  PAINT_FGA_RATE           450 non-null

In [6]:
features = players.drop(["PLAYER_ID", "PLAYER_NAME"], axis=1).columns.to_list()

print(players[features].isna().sum())
print(np.isinf(players[features]).sum())
print(players[features].std())

USG_RATE                   0
TOUCHES_PER_100            0
POT_AST_PER_100            0
DRIVES_PER_100             0
CATCH_SHOOT_FGA_PER_100    0
PULL_UP_FGA_PER_100        0
PAINT_TOUCHES_PER_100      0
SCREEN_AST_PER_100         0
DEFLECTIONS_PER_100        0
STL_PCT                    0
BLK_PCT                    0
BOXOUTS_PER_100            0
PAINT_FGA_RATE             0
MIDRANGE_FGA_RATE          0
dtype: int64
USG_RATE                   0
TOUCHES_PER_100            0
POT_AST_PER_100            0
DRIVES_PER_100             0
CATCH_SHOOT_FGA_PER_100    0
PULL_UP_FGA_PER_100        0
PAINT_TOUCHES_PER_100      0
SCREEN_AST_PER_100         0
DEFLECTIONS_PER_100        0
STL_PCT                    0
BLK_PCT                    0
BOXOUTS_PER_100            0
PAINT_FGA_RATE             0
MIDRANGE_FGA_RATE          0
dtype: int64
USG_RATE                    0.053039
TOUCHES_PER_100            18.576337
POT_AST_PER_100             4.811384
DRIVES_PER_100              8.052690
CATCH_SHOOT_FG

In [8]:
X = players[features].copy()
X.shape

(450, 14)

In [9]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns = features, index = players.index)

print(X_scaled_df.mean().round(2))
print(X_scaled_df.std().round(2))
X_scaled_df.head()

USG_RATE                   0.0
TOUCHES_PER_100           -0.0
POT_AST_PER_100           -0.0
DRIVES_PER_100            -0.0
CATCH_SHOOT_FGA_PER_100    0.0
PULL_UP_FGA_PER_100       -0.0
PAINT_TOUCHES_PER_100      0.0
SCREEN_AST_PER_100         0.0
DEFLECTIONS_PER_100       -0.0
STL_PCT                   -0.0
BLK_PCT                    0.0
BOXOUTS_PER_100            0.0
PAINT_FGA_RATE             0.0
MIDRANGE_FGA_RATE         -0.0
dtype: float64
USG_RATE                   1.0
TOUCHES_PER_100            1.0
POT_AST_PER_100            1.0
DRIVES_PER_100             1.0
CATCH_SHOOT_FGA_PER_100    1.0
PULL_UP_FGA_PER_100        1.0
PAINT_TOUCHES_PER_100      1.0
SCREEN_AST_PER_100         1.0
DEFLECTIONS_PER_100        1.0
STL_PCT                    1.0
BLK_PCT                    1.0
BOXOUTS_PER_100            1.0
PAINT_FGA_RATE             1.0
MIDRANGE_FGA_RATE          1.0
dtype: float64


,USG_RATE,TOUCHES_PER_100,POT_AST_PER_100,DRIVES_PER_100,CATCH_SHOOT_FGA_PER_100,PULL_UP_FGA_PER_100,PAINT_TOUCHES_PER_100,SCREEN_AST_PER_100,DEFLECTIONS_PER_100,STL_PCT,BLK_PCT,BOXOUTS_PER_100,PAINT_FGA_RATE,MIDRANGE_FGA_RATE
0,-0.942783,-0.505670,-0.645524,-0.910562,1.267531,-0.047780,-0.939171,-0.539604,-0.775523,-1.405032,-1.071490,-0.387096,-2.060258,-0.577309
1,0.189714,-0.098932,0.227509,0.894536,-0.678260,-0.313349,-0.502151,-0.699726,-0.839722,-0.629881,-0.898473,-0.946035,0.755379,-0.862930
2,0.529463,0.205146,-0.159975,-0.621380,-0.136145,0.246436,0.877706,0.090241,-1.566525,-0.839818,-0.462470,-0.469305,-0.069969,0.656446
3,-0.489784,-0.395123,-0.301470,-0.113743,1.304167,-0.149876,-1.029892,-0.665542,-0.738104,0.080673,-0.946918,-0.667978,-1.106922,-0.261020
4,0.095339,0.073847,-0.734166,0.302626,0.892055,0.165172,-0.723176,-0.345877,-1.231835,-0.969010,0.056580,-0.709764,-0.699021,0.445300


In [12]:
test_kmeans = KMeans(n_clusters=5, random_state=28, n_init=20)
test_labels = test_kmeans.fit_predict(X_scaled)
pd.Series(test_labels).value_counts().sort_index()

0    159
1    120
2     41
3     73
4     57
Name: count, dtype: int64

In [16]:
k_values = range(2, 13)
inertias = []
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=28, n_init=20)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

k_results = pd.DataFrame({"k": k_values, "inertia": inertias, "silhouette_score": silhouette_scores})
k_results

,k,inertia,silhouette_score
0,2,4525.546387,0.329562
1,3,3395.213753,0.273125
2,4,3011.783635,0.224017
3,5,2785.833088,0.219348
4,6,2603.666369,0.192002
5,7,2446.801822,0.170076
6,8,2340.747802,0.167727
7,9,2239.782644,0.165657
8,10,2153.634834,0.165386
9,11,2066.158676,0.169200
